# 3D plots

Anton Antonov  
April 2026  
November 2026

---

## Setup

In [ ]:
use WWW::HorizonsEphemerisSystem;
use JavaScript::D3;

use Data::Generators;
use Data::Reshapers;
use Data::Summarizers;

In [ ]:
%%javascript
require.config({
  paths: {
    d3: "https://d3js.org/d3.v7.min",
    d3_3d: "https://unpkg.com/d3-3d@2.0.2/build/d3-3d"
  }
});

require(["d3", "d3_3d"], function(d3, d3_3d) {
  window.d3 = d3;
  window.d33d = d3_3d || window.d33d || {};
  console.log("d3:", d3.version, "d3-3d exports:", Object.keys(window.d33d));
});

In [ ]:
my $background = '#1F1F1F';
my $title-color = 'Ivory';

In [ ]:
#%js
js-d3-list-line-plot(rand xx 30, :$background)

---

## 3D random points

In [ ]:
my @generators =
    {&random-variate(NormalDistribution.new(1, 1), $_)}, 
    {&random-variate(NormalDistribution.new(1, 1.4), $_)}, 
    {&random-variate(NormalDistribution.new(4, 3), $_)},
;

my @data = random-tabular-dataset(300, <x y z>, :@generators);

deduce-type(@data)

In [ ]:
sink records-summary(@data)

In [ ]:
#%js
js-d3-list-point-plot3d(@data, :$background, :!axes, :boxed)

---

## 3D random points with groups

In [ ]:
my @generators =
    {&random-variate(NormalDistribution.new(1, 1), $_)}, 
    {&random-variate(NormalDistribution.new(1, 1.4), $_)}, 
    {&random-variate(NormalDistribution.new(4, 3), $_)},
    { ('a' xx $_).List }
;

my @data1 = random-tabular-dataset(500, <x y z group>, :@generators);

@generators =
    {&random-variate(NormalDistribution.new(5, 1), $_)}, 
    {&random-variate(NormalDistribution.new(7, 3.4), $_)}, 
    {&random-variate(NormalDistribution.new(6, 2), $_)},
    { ('b' xx $_).List }
;

my @data2 = random-tabular-dataset(500, <x y z group>, :@generators);

my @data = |@data1, |@data2;

deduce-type(@data)

In [ ]:
sink records-summary(@data)

In [ ]:
#%js
my $box-ratios = [1, 1, 0.4];
js-d3-list-point-plot3d(
    @data, 
    :$title-color,
    x-label => 'X',
    x-label-color => 'red',
    x-label-font-size => 16,
    y-label => 'Y',
    y-label-color => 'blue',
    z-label => 'Z',
    z-label-color => 'green',
    :$background, 
    :$box-ratios, 
    :axes,
    :!boxed,
    :legends,
    :1200width,
    :700height
)

In [ ]:
#%js
js-d3-list-point-plot3d(
    @data, 
    :$background, 
    box-ratios => Whatever, 
    color-scheme => 'Category10',
    :!axes, 
    :boxed, 
    :ticks, 
    title => 'Random points', 
    :!legends
)

---

## 3D line plot

In [ ]:
my @data = (1...4).map( -> $r { (0, π/30 ... 2*π + π/30).map(-> $t { <x y z group>.Array Z=> [$r * sin($t), $r * cos($t), cos($r*$t), $r.Str] })».Hash })».List;
@data .= map(*.Slip);
deduce-type(@data)

In [ ]:
#%js
my @color-scheme = "#a4a74e", "#f28e2b", "#e15759", "#76b7b2", "#59a14f", "#edc948", "#b07aa1", "#ff9da7", "#9c755f", "#bab0ab";
js-d3-list-line-plot3d(
    @data,
    :$background,
    :@color-scheme
)

---

## 3D surfaces 

Irregular data:

In [ ]:
#% js
js-d3-list-plot3d(
    [[0, 0, 1], [1, 0, 0], [0, 1, 0]], 
    :$background, 
    :$title-color,
)

Regular data (a matrix of Z-values):

In [ ]:
#% js
js-d3-list-plot3d(
    (^6).rotor(4 => -3),
    :$background
)

Larger irregular data:

In [ ]:
my $n = 20;
my @data = ((0, pi/$n ... pi) X (0, pi/$n ... pi)).map({ [|$_, sin( $_.tail ** 2 + $_.head )]».Num });
deduce-type(@data)

In [ ]:
#% js
js-d3-list-plot3d(
    @data,
    :$background, 
    :$title-color,
    :!axes,
    #view-point => [1.3, -2.4, 2.0],
    :16scale
)

Larger regular data:

In [ ]:
my $n = 20;
my @matrix = ((0, pi/($n-1) ... pi) X (0, pi/($n-1) ... pi)).map({ sin( $_.tail ** 2 + $_.head ) }).rotor($n);
deduce-type(@matrix)

In [ ]:
#% js
js-d3-list-plot3d(
    @matrix,
    :$background, 
    :$title-color,
    :!axes,
    :16scale
)

### Multiple datasets

Long form datasets:

In [ ]:
#% js

my @data1 = random-real([-1, 1], 1200).map({ <x y z group> Z=> [$^x, $^y, $^x ** 2 - $^y ** 2, 'first'] })».Hash;
my @data2 = random-real([-1, 1], 1200).map({ <x y z group> Z=> [$^x, $^y, $^y ** 2 - $^x ** 2, 'second'] })».Hash;
my @data = |@data1, |@data2;

js-d3-list-plot3d(@data, :$background, :$title-color, box-ratios => [1,1,0.5], :!wireframe, :legend)